<a href="https://colab.research.google.com/github/MohamedAamer01/otto-recommender-sys/blob/main/notebooks/08_final_b4star.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Environnement**

In [1]:
import os, sys, time, pickle
import numpy as np, pandas as pd
from google.colab import drive, userdata
drive.mount("/content/drive")

BASE    = "/content/drive/MyDrive/OTTO_Recommender_System"
NPY_DIR = f"{BASE}/data/processed"
SRC     = f"{BASE}/src"
os.makedirs(SRC, exist_ok=True)
sys.path.insert(0, SRC)

Mounted at /content/drive


Ecriture du modele

In [2]:
%%writefile /content/drive/MyDrive/OTTO_Recommender_System/src/otto_core.py
"""otto_core.py - fonctions essentielles du projet OTTO (testées contre un calcul brut ou la version notebook)."""
import time
import numpy as np

WEIGHTS = {"clicks": 0.10, "carts": 0.30, "orders": 0.60}
TYPE2ID = {"clicks": 0, "carts": 1, "orders": 2}


# ---------------------------------------------------------------- co-visitation
def covisit_block(aid_b, ts_b, sizes_b, K, window, tmax=None):
    """Paires ordonnées (i, j) des K derniers événements (ts < tmax) de chaque session.
    Retourne (clés triées, compteurs). clé = i * 2**21 + j."""
    S = len(sizes_b)
    sess = np.repeat(np.arange(S), sizes_b)
    if tmax is not None:                                   # anti-fuite : on retire les événements >= tmax
        keep = ts_b < tmax
        sess, aid_b, ts_b = sess[keep], aid_b[keep], ts_b[keep]
    sz = np.bincount(sess, minlength=S)
    starts = np.cumsum(sz) - sz
    pos_end = sz[sess] - 1 - (np.arange(len(sess)) - starts[sess])      # 0 = dernier événement
    sel = pos_end < K
    A = np.full((S, K), -1, dtype=np.int64)
    Tm = np.zeros((S, K), dtype=np.int64)
    A[sess[sel], pos_end[sel]] = aid_b[sel]
    Tm[sess[sel], pos_end[sel]] = ts_b[sel]
    ai, aj = A[:, :, None], A[:, None, :]
    ok = (ai >= 0) & (aj >= 0) & (ai != aj) & (np.abs(Tm[:, :, None] - Tm[:, None, :]) <= window)
    keys = (np.broadcast_to(ai, ok.shape)[ok] << 21) | np.broadcast_to(aj, ok.shape)[ok]
    return np.unique(keys, return_counts=True)


def merge(parts):
    k = np.concatenate([p[0] for p in parts]); c = np.concatenate([p[1] for p in parts])
    u, inv = np.unique(k, return_inverse=True)
    return u, np.bincount(inv, weights=c).astype(np.int64)


def build_topn(aid_arr, ts_arr, sizes_arr, cum, n_items, K=10, window=86400, tmax=None,
               N=50, P=4, BLOCK=50_000):
    """Table des N meilleurs voisins par article : nbr_j[i] = voisins, nbr_c[i] = compteurs."""
    nbr_j = np.full((n_items, N), -1, dtype=np.int32)
    nbr_c = np.zeros((n_items, N), dtype=np.float32)
    n_sess, MASK = len(sizes_arr), (1 << 21) - 1
    for p in range(P):
        t0, parts = time.time(), []
        for s in range(0, n_sess, BLOCK):
            e = min(s + BLOCK, n_sess); a, z = cum[s], cum[e]
            k, c = covisit_block(aid_arr[a:z], ts_arr[a:z], sizes_arr[s:e], K, window, tmax)
            m = ((k >> 21) % P) == p
            parts.append((k[m], c[m]))
            if len(parts) >= 20:
                parts = [merge(parts)]
        keys, counts = merge(parts)
        i = (keys >> 21).astype(np.int64); j = (keys & MASK).astype(np.int32)
        order = np.lexsort((-counts, i))
        i, j, counts = i[order], j[order], counts[order]
        rank = np.arange(len(i)) - np.searchsorted(i, i, side="left")
        keep = rank < N
        nbr_j[i[keep], rank[keep]] = j[keep]
        nbr_c[i[keep], rank[keep]] = counts[keep]
        print(f"partition {p+1}/{P} : {len(keys):,} paires distinctes | {time.time()-t0:.0f}s")
        del keys, counts, i, j, order, parts
    return nbr_j, nbr_c


# ---------------------------------------------------------------- popularité et listes
def top_lists(aid_arr, type_arr, n_items, mask=None, k=40):
    """Top-k articles par type (clicks, carts, orders), sur les événements où mask est vrai."""
    m = np.ones(len(aid_arr), bool) if mask is None else mask
    return {t: np.argsort(-np.bincount(aid_arr[m & (type_arr == i)], minlength=n_items),
                          kind="stable")[:k].tolist() for t, i in TYPE2ID.items()}


def topup(base, top, n=20):
    lst = list(base); s = set(lst)
    for x in top:
        if len(lst) == n: break
        if x not in s: lst.append(x)
    return lst


def lists_b4star(seen, tops, nj, nc, n_seed=10, gamma=0.5, row_norm=True, n=20):
    """B4* : articles de la session (récents d'abord), puis candidats de co-visitation, puis popularité par type.
    seen : articles distincts de la session, du plus récent au plus ancien."""
    base = seen[:n]; sset = set(base)
    if len(base) < n:
        sd = np.asarray(seen[:n_seed])
        cs = nc[sd].astype(np.float64)
        if row_norm:
            cs = cs / np.maximum(cs.sum(axis=1, keepdims=True), 1)
        cs = cs * (gamma ** np.arange(len(sd)))[:, None]
        js = nj[sd].ravel(); cs = cs.ravel(); m = js >= 0
        if m.any():
            u, inv = np.unique(js[m], return_inverse=True)
            sc = np.bincount(inv, weights=cs[m])
            order = np.argsort(-sc, kind="stable")
            base = base + [int(x) for x in u[order] if int(x) not in sset][:n - len(base)]
    return {t: topup(base, tops[t], n) for t in WEIGHTS}


def write_submission(path, ids, t_aid, t_size, tops, nj, nc, **kw):
    """Écrit le fichier de soumission en flux : 3 lignes par session (clicks, carts, orders)."""
    off = np.concatenate([[0], np.cumsum(t_size, dtype=np.int64)])
    t0 = time.time()
    with open(path, "w") as f:
        f.write("session_type,labels\n")
        for i in range(len(ids)):
            seen = list(dict.fromkeys(map(int, t_aid[off[i]:off[i + 1]][::-1])))
            r = lists_b4star(seen, tops, nj, nc, **kw)
            for t in WEIGHTS:
                f.write(f"{ids[i]}_{t},{' '.join(map(str, r[t]))}\n")
            if (i + 1) % 500_000 == 0:
                print(f"{i+1:,} sessions ({time.time()-t0:.0f}s)")


# ---------------------------------------------------------------- validation et métrique
def split_labels(suf_aid, suf_type):
    clicks = suf_aid[suf_type == 0]
    return {"clicks": {int(clicks[0])} if len(clicks) else set(),
            "carts": set(map(int, suf_aid[suf_type == 1])),
            "orders": set(map(int, suf_aid[suf_type == 2]))}


def per_session(make, GT, k=20):
    sids = list(GT)
    H = {t: np.zeros(len(sids)) for t in WEIGHTS}; D = {t: np.zeros(len(sids)) for t in WEIGHTS}
    for i, sid in enumerate(sids):
        lists = make(sid)
        for t in WEIGHTS:
            gt = GT[sid][t]
            if gt:
                H[t][i] = len(set(lists[t][:k]) & gt); D[t][i] = min(k, len(gt))
    return H, D


def score_from(H, D, idx=None):
    out = {}
    for t in WEIGHTS:
        h = H[t] if idx is None else H[t][idx]; d = D[t] if idx is None else D[t][idx]
        out[t] = h.sum() / d.sum()
    out["score"] = sum(WEIGHTS[t] * out[t] for t in WEIGHTS)
    return out


def boot_diff_all(HA, HB, D, B=100, seed=0):
    r = np.random.default_rng(seed); n = len(next(iter(D.values())))
    diffs = {k: [] for k in list(WEIGHTS) + ["score"]}
    for _ in range(B):
        idx = r.integers(0, n, n); a = score_from(HA, D, idx); b = score_from(HB, D, idx)
        for k in diffs: diffs[k].append(b[k] - a[k])
    return {k: np.percentile(v, [2.5, 97.5]) for k, v in diffs.items()}

Writing /content/drive/MyDrive/OTTO_Recommender_System/src/otto_core.py


**test unitaire du module**

In [3]:
import importlib, tempfile
from collections import Counter
import otto_core; importlib.reload(otto_core)
from otto_core import *

rng = np.random.default_rng(3)
S = 600; sizes = rng.integers(2, 25, S).astype(np.int32); Nn = int(sizes.sum())
aid = rng.integers(0, 60, Nn).astype(np.int32)
ts  = np.concatenate([np.sort(rng.integers(0, 300, n)) for n in sizes]).astype(np.int32)
ty  = rng.choice([0, 1, 2], Nn, p=[.9, .08, .02]).astype(np.int8)
cum_t = np.concatenate([[0], np.cumsum(sizes, dtype=np.int64)])

def brute(K, W, tm):
    C, p = Counter(), 0
    for n in sizes:
        a, t = aid[p:p+n], ts[p:p+n]; p += n
        m = t < tm; a, t = a[m][-K:], t[m][-K:]
        for x in range(len(a)):
            for y in range(len(a)):
                if x != y and a[x] != a[y] and abs(int(t[x]) - int(t[y])) <= W:
                    C[(int(a[x]), int(a[y]))] += 1
    return C

k, c = covisit_block(aid, ts, sizes, 8, 100, 250)
got = {(int(x >> 21), int(x & ((1 << 21) - 1))): int(v) for x, v in zip(k, c)}
assert got == dict(brute(8, 100, 250)); print("covisit_block conforme au calcul brut")

a1 = build_topn(aid, ts, sizes, cum_t, 60, K=8, window=100, tmax=250, N=5, P=4, BLOCK=100)
a2 = build_topn(aid, ts, sizes, cum_t, 60, K=8, window=100, tmax=250, N=5, P=1, BLOCK=100)
assert np.array_equal(a1[0], a2[0]) and np.allclose(a1[1], a2[1]); print("partitions conformes")

tops_t = top_lists(aid, ty, 60, ts < 250)
ids = np.arange(900, 905); t_size = np.array([2, 3, 1, 4, 30], dtype=np.int32)
t_aid = rng.integers(0, 60, t_size.sum()).astype(np.int32)
path = tempfile.mktemp(suffix=".csv"); write_submission(path, ids, t_aid, t_size, tops_t, *a1)
back = pd.read_csv(path); assert back.shape == (15, 2)
assert back["labels"].map(lambda s: len(s.split()) == 20 and len(set(s.split())) == 20).all()
print("fichier de soumission conforme")

GT_t = {s: split_labels(aid[cum_t[s]+1:cum_t[s+1]], ty[cum_t[s]+1:cum_t[s+1]]) for s in range(50)}
H_, D_ = per_session(lambda s: {t: list(GT_t[s][t])[:20] for t in WEIGHTS}, GT_t)
assert abs(score_from(H_, D_)["score"] - 1) < 1e-12; print("oracle = 1.0")

covisit_block conforme au calcul brut
partition 1/4 : 883 paires distinctes | 0s
partition 2/4 : 880 paires distinctes | 0s
partition 3/4 : 879 paires distinctes | 0s
partition 4/4 : 884 paires distinctes | 0s
partition 1/1 : 3,526 paires distinctes | 0s
partitions conformes
fichier de soumission conforme
oracle = 1.0


**chargement, et non-régression sur le dev**

In [4]:
N, N_SESSIONS = 216_716_096, 12_899_779
aid_arr   = np.load(f"{NPY_DIR}/aid.npy");  ts_arr   = np.load(f"{NPY_DIR}/ts.npy")
type_arr  = np.load(f"{NPY_DIR}/type.npy"); sizes_arr = np.load(f"{NPY_DIR}/sizes.npy")
assert len(aid_arr) == N and len(sizes_arr) == N_SESSIONS and sizes_arr.sum() == N
cum = np.concatenate([[0], np.cumsum(sizes_arr, dtype=np.int64)])
n_items = int(aid_arr.max()) + 1

with open(f"{NPY_DIR}/dev_validation.pkl", "rb") as f:
    dev = pickle.load(f)
T, GT, prefix_len = dev["T"], dev["GT"], dev["prefix_len"]
nj = np.load(f"{NPY_DIR}/covisit_K10_T_nbr_j.npy"); nc = np.load(f"{NPY_DIR}/covisit_K10_T_nbr_c.npy")
tops_T = top_lists(aid_arr, type_arr, n_items, ts_arr < T)

def seen_dev(sid):
    a, k = cum[sid], prefix_len[sid]
    return list(dict.fromkeys(map(int, aid_arr[a:a + k][::-1])))

H, D = per_session(lambda s: lists_b4star(seen_dev(s), tops_T, nj, nc), GT)
r = score_from(H, D)
print({k: round(v, 4) for k, v in r.items()}, "(attendu : score 0.5497)")
assert abs(r["score"] - 0.5497) < 5e-4, "le module ne redonne pas B4* : ne pas soumettre"

{'clicks': np.float64(0.5134), 'carts': np.float64(0.4042), 'orders': np.float64(0.6285), 'score': np.float64(0.5497)} (attendu : score 0.5497)


**Table finale sur tout le train, et popularité**

In [5]:
t0 = time.time()
nbr_j_all, nbr_c_all = build_topn(aid_arr, ts_arr, sizes_arr, cum, n_items,
                                  K=10, window=86400, tmax=None, N=50, P=4)
print(f"Construction : {(time.time()-t0)/60:.1f} min")
print(f"Articles avec au moins 1 voisin : {(nbr_j_all[:, 0] >= 0).mean()*100:.1f} % (avant T : 91.5 %)")
np.save(f"{NPY_DIR}/covisit_K10_ALL_nbr_j.npy", nbr_j_all)
np.save(f"{NPY_DIR}/covisit_K10_ALL_nbr_c.npy", nbr_c_all)

tops_all = top_lists(aid_arr, type_arr, n_items)                       # popularité : tout le train
print("Top-5 orders :", tops_all["orders"][:5])

partition 1/4 : 31,642,272 paires distinctes | 65s
partition 2/4 : 31,531,515 paires distinctes | 63s
partition 3/4 : 31,589,946 paires distinctes | 64s
partition 4/4 : 31,407,527 paires distinctes | 67s
Construction : 4.3 min
Articles avec au moins 1 voisin : 95.3 % (avant T : 91.5 %)
Top-5 orders : [231487, 166037, 1733943, 1445562, 1022566]


# **données de test, vérifications**

In [6]:
import zipfile
os.environ["KAGGLE_API_TOKEN"] = userdata.get("KAGGLE_API_TOKEN")
DATA_DIR = "/content/otto_data"; os.makedirs(DATA_DIR, exist_ok=True)
!pip install -q --upgrade kaggle
!kaggle competitions download -c otto-recommender-system -f sample_submission.csv -p /content/otto_data
zp = f"{DATA_DIR}/sample_submission.csv.zip"
if os.path.exists(zp):
    with zipfile.ZipFile(zp) as z: z.extractall(DATA_DIR)

t_aid  = np.load(f"{NPY_DIR}/test_aid.npy")
t_size = np.load(f"{NPY_DIR}/test_sizes.npy")
ids    = np.load(f"{NPY_DIR}/test_session_ids.npy")
sample = pd.read_csv(f"{DATA_DIR}/sample_submission.csv")

assert len(t_size) == len(ids) == 1_671_803 and t_size.sum() == len(t_aid) == 6_928_123
assert t_aid.max() < n_items                                           # tous les aid du test existent dans le train
first = sample["session_type"].iloc[::3].str.split("_").str[0].astype("int64").values
assert np.array_equal(first, ids), "ordre différent de sample_submission"
print("Test et ordre des identifiants : OK |", len(sample), "lignes attendues")

100% 12.6M/12.6M [00:01<00:00, 7.75MB/s]

Test et ordre des identifiants : OK | 5015409 lignes attendues


**génération, vérification du format**

In [7]:
import shutil
sub_path = "/content/submission_B4star.csv"
write_submission(sub_path, ids, t_aid, t_size, tops_all, nbr_j_all, nbr_c_all)
print(f"Taille : {os.path.getsize(sub_path)/1e6:.0f} MB")

back = pd.read_csv(sub_path, usecols=["session_type"])
assert back["session_type"].equals(sample["session_type"]), "ordre ou clés différents"
chk = pd.read_csv(sub_path, nrows=300_000)
assert chk["labels"].map(lambda s: len(s.split()) == 20 and len(set(s.split())) == 20).all()
print("Format OK :", back.shape[0], "lignes (attendu 5 015 409)")

os.makedirs(f"{BASE}/submissions", exist_ok=True)
shutil.copy(sub_path, f"{BASE}/submissions/submission_B4star.csv")

500,000 sessions (67s)
1,000,000 sessions (125s)
1,500,000 sessions (181s)
Taille : 818 MB
Format OK : 5015409 lignes (attendu 5 015 409)


'/content/drive/MyDrive/OTTO_Recommender_System/submissions/submission_B4star.csv'

**Soumission**

In [8]:
!kaggle competitions submit -c otto-recommender-system -f /content/submission_B4star.csv -m "B4*: session + covisit (K=10, 24h, top-50, gamma 0.5, row-norm) + popularity"

100% 780M/780M [00:32<00:00, 25.1MB/s]
Submission ref: 58794035
99 submissions remaining today.
Successfully submitted to OTTO – Multi-Objective Recommender System

In [10]:
!kaggle competitions submissions -c otto-recommender-system

     ref  fileName               date                        description                                                                   status                     publicScore  privateScore  
--------  ---------------------  --------------------------  ----------------------------------------------------------------------------  -------------------------  -----------  ------------  
58794035  submission_B4star.csv  2026-11-23 20:25:28.467000  B4*: session + covisit (K=10, 24h, top-50, gamma 0.5, row-norm) + popularity  SubmissionStatus.COMPLETE  0.56353      0.56271       
58723135  submission_B3.csv      2026-11-22 13:27:05.023000  B3: session items (recent first) + popularity per type                        SubmissionStatus.COMPLETE  0.47911      0.48035       
58722530  submission_B3.csv      2026-11-22 13:11:06.330000  B3: session items (recent first) + popularity per type                        SubmissionStatus.COMPLETE  0.47911      0.48035       
